# Kernel Regression 

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i,y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

is the feature vector and

$$
y_i \in \mathbb{R}
$$

is the target value.

Kernel regression is a **non-parametric method**, meaning it does not assume a fixed functional form for the relationship between $x$ and $y$.

---

# 2. Distance Between Data Points

To measure similarity between observations, we compute the squared Euclidean distance between a test point $x$ and a training point $x_i$:

$$
d(x,x_i)^2 = \|x-x_i\|^2
$$

In matrix form for test data $X_{test}$ and training data $X_{train}$:

$$
D =
\|X_{test}\|^2 +
\|X_{train}\|^2 -
2X_{test}X_{train}^T
$$

Each element $D_{ij}$ represents the squared distance between test point $i$ and training point $j$.

---

# 3. Kernel Function

Kernel regression assigns weights to training points using a **kernel function**.

The Gaussian (RBF) kernel is

$$
K(x,x_i)=
\exp
\left(
-\frac{\|x-x_i\|^2}{2h^2}
\right)
$$

where $h$ is the **bandwidth parameter**.

- Large $h$ produces smoother predictions and more global averaging.
- Small $h$ gives more local influence and predictions that follow the training data closely.

---

# 4. Kernel Matrix

For all test and training points we compute the kernel matrix

$$
K_{ij} =
\exp
\left(
-\frac{D_{ij}}{2h^2}
\right)
$$

where

$$
D_{ij} = \|x_i^{test}-x_j^{train}\|^2
$$

Thus

$$
K \in \mathbb{R}^{N_{test} \times N_{train}}
$$

Each entry represents the similarity between a test point and a training point.

---

# 5. Kernel Regression Prediction

The prediction for a test point $x$ is a **weighted average of training targets**:

$$
\hat{y}(x)=
\frac{\sum_{i=1}^{N} K(x,x_i)y_i}
{\sum_{i=1}^{N} K(x,x_i)}
$$

The numerator computes the weighted contribution of training outputs.

The denominator normalizes the weights.

---

# 6. Matrix Form of Prediction

Let

$$
K = K(X_{test},X_{train})
$$

Then predictions for all test samples are

$$
\hat{y} =
\frac{Ky}{\sum K}
$$

where the division is performed **row-wise**.

More explicitly

$$
\hat{y}_i =
\frac{\sum_{j=1}^{N} K_{ij}y_j}
{\sum_{j=1}^{N} K_{ij}}
$$

---

# 7. Role of the Bandwidth Parameter

The bandwidth $h$ determines how much influence nearby points have.

- **Small $h$** gives a highly local model and predictions may become noisy.
- **Large $h$** produces smoother predictions but may oversmooth the data.

Thus $h$ controls the **bias–variance trade-off**.

---

# 8. Summary

Kernel regression:

- Uses similarity between data points instead of explicit parameters
- Computes weights using the Gaussian kernel

$$
K(x,x_i)=
\exp
\left(
-\frac{\|x-x_i\|^2}{2h^2}
\right)
$$

Predictions are obtained using 

$$
\hat{y}(x)=
\frac{\sum_{i=1}^{N} K(x,x_i)y_i}
{\sum_{i=1}^{N} K(x,x_i)}
$$

This produces a smooth regression function based on **local averaging of nearby training observations**.

In [1]:
class KernelRegression:
    def __init__(self,h =0.1):
        self.h = h
        self.K = None

    def kernel_func(self,X_test,X_train):
        D = np.sum(X_test*X_test,axis=1,keepdims=True) + np.sum(X_train*X_train,axis=1) - 2 * X_test @ X_train.T
        h = max(self.h,0.1)
        two_h_squared = 2* (h**2)

        
        D = np.maximum(D,0)
        
        return np.exp(-D/ two_h_squared)

    def predict(self, X_train , X_test , y_train):
        X_train = np.asarray(X_train)
        X_test = np.asarray(X_test)
        y_train = np.asarray(y_train)
        y_train = y_train.reshape(-1)

        if X_train.ndim==1:
            X_train = X_train.reshape(-1,1)
        if X_test.ndim==1:
            X_test = X_test.reshape(-1,1)


        self.K = self.kernel_func(X_test,X_train)
        numerator = self.K @ y_train
        denominator = np.sum(self.K,axis=1) + 1e-10


        predictions = numerator / denominator

        return predictions
      

## 1. Create a Simple Dataset

We construct a small dataset with three training points.

$$
X_{train} =
\begin{bmatrix}
0 \\
1 \\
3
\end{bmatrix}
$$

and corresponding targets

$$
y =
\begin{bmatrix}
1 \\
2 \\
4
\end{bmatrix}
$$

We evaluate predictions at the test point

$$
x_{test} = 1.5
$$

This point lies closer to the training point $x=1$ than the others.

In [2]:
import numpy as np

X_train = np.array([[0],[1],[3]])
y_train = np.array([1,2,4])

X_test = np.array([[1.5]])

---

## 2. Distance Matrix

Kernel regression first computes the squared Euclidean distance between the test point and each training point.

$$
D(x,x_i) = (x-x_i)^2
$$

In [3]:
D = (X_test - X_train.T)**2
print("Distance Matrix:")
print(D)

Distance Matrix:
[[2.25 0.25 2.25]]


---

## 3. Kernel Matrix (Small $h$)

Using the Gaussian kernel

$$
K(x,x_i)=
\exp
\left(
-\frac{(x-x_i)^2}{2h^2}
\right)
$$

When $h$ is small, the kernel assigns **very large weight to nearby points** and **very small weight to distant points**.

In [4]:
h_small = 0.3

K_small = np.exp(-D/(2*h_small**2))

print("Kernel Matrix (small h):")
print(K_small)

Kernel Matrix (small h):
[[3.72665317e-06 2.49352209e-01 3.72665317e-06]]


**Observation**

The closest point $x = 1$ gets very high weight.  

Distant points contribute almost nothing.  

Thus, the prediction will be close to $y = 2$.

---

## 4. Kernel Matrix (Large $h$)

When the bandwidth $h$ is large, the kernel decays slowly and distant points still contribute significant weight.

In [5]:
h_large = 20.0

K_large = np.exp(-D/(2*h_large**2))

print("Kernel Matrix (large h):")
print(K_large)

Kernel Matrix (large h):
[[0.99719145 0.99968755 0.99719145]]


**Observation**

All points now receive similar weights.  

Thus, the prediction becomes closer to the global average of the targets.

---

## 5. Kernel Regression Prediction

Kernel regression computes predictions as

$$
\hat{y}(x)=
\frac{\sum_i K(x,x_i)y_i}
{\sum_i K(x,x_i)}
$$

In [6]:
pred_small = (K_small @ y_train) / np.sum(K_small)
pred_large = (K_large @ y_train) / np.sum(K_large)

print("Prediction (small h):", pred_small)
print("Prediction (large h):", pred_large)

Prediction (small h): [2.00001494]
Prediction (large h): [2.33305544]


---

## 6. Interpretation

Small $h$:

- Kernel weights are concentrated around nearby points.
- The model behaves like a **local regression**.
- Predictions follow the training data closely.

Large $h$:

- Kernel weights become similar across all points.
- The model behaves like **global averaging**.
- Predictions become smoother but may lose local structure.

Thus the bandwidth $h$ controls the **bias–variance trade-off** in kernel regression.

In [7]:
for h in [0.3, 20.0]:
    model = KernelRegression(h=h)
    y_pred = model.predict(X_train, X_test, y_train)
    
    print(f"\nKernel Regression with h = {h}")
    print(f"Kernel weights K for test point {X_test.flatten()}: {model.K}")
    print(f"Predicted value: {y_pred.flatten()[0]}")

print(f"\nGlobal average of y_train: {np.mean(y_train)}")


Kernel Regression with h = 0.3
Kernel weights K for test point [1.5]: [[3.72665317e-06 2.49352209e-01 3.72665317e-06]]
Predicted value: 2.0000149440897514

Kernel Regression with h = 20.0
Kernel weights K for test point [1.5]: [[0.99719145 0.99968755 0.99719145]]
Predicted value: 2.3330554398334438

Global average of y_train: 2.3333333333333335


---